In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.feature_selection import  VarianceThreshold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectFromModel, SequentialFeatureSelector
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.feature_selection import mutual_info_classif
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV
from scipy.stats import randint
from skopt.space import Integer, Categorical
from sklearn.linear_model import LogisticRegression
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_validate
import xgboost as xgb
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

In [4]:
X_train = pd.read_csv("artifical_train_data.csv")
y_train = pd.read_csv("artifical_train_labels.csv")
y_train = y_train['Class']

In [5]:
random_state = 111111

# SVM + z f_classif

In [8]:
pipeline_svm_1 = Pipeline([
    ('pre', SelectKBest(score_func=f_classif)),
    ('model', SVC(probability=True))
])

param_grid = {
    'pre__k': range(5,20),
    'model__C': np.logspace(-1,4,6)
}

svm_grid_2 = GridSearchCV(
    estimator=pipeline_svm_1,
    param_grid=param_grid,
    n_jobs=-1,
    cv=5,
    scoring='balanced_accuracy'
)

svm_grid_2.fit(X_train,y_train)

,estimator,Pipeline(step...ility=True))])
,param_grid,"{'model__C': array([1.e-01...e+03, 1.e+04]), 'pre__k': range(5, 20)}"
,scoring,'balanced_accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,score_func,<function f_c...001E9F2761800>


In [10]:
# 1. Wyciągamy najlepszy rurociąg (pipeline) z przeszukiwania
best_pipeline = svm_grid_2.best_estimator_

# 2. Dobieramy się do kroku 'pre' (SelectKBest) w najlepszym rurociągu
best_selector = best_pipeline.named_steps['pre']

# 3. Pobieramy maskę logiczną (True dla wybranych zmiennych, False dla odrzuconych)
selected_mask = best_selector.get_support()

# 4. Nakładamy maskę na nazwy kolumn z X_train
selected_features = X_train.columns[selected_mask]

print("Wybrane zmienne:", selected_features.tolist())

Wybrane zmienne: ['V3', 'V8', 'V9', 'V10', 'V42', 'V48', 'V51', 'V58', 'V65', 'V67', 'V83', 'V91']


# Random Forest Importance

In [11]:
rf = RandomForestClassifier(random_state=random_state)
rf.fit(X_train,y_train)
importances = rf.feature_importances_
feature_importance = pd.Series(importances, index=X_train.columns)
feature_importance.sort_values(ascending=False).head(10)

V51    0.041671
V3     0.030779
V42    0.029084
V58    0.025233
V91    0.024051
V67    0.023244
V10    0.022790
V83    0.021602
V50    0.020907
V57    0.019945
dtype: float64

In [14]:
corr_matrix = X_train.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
X_train_final = X_train.drop(columns=to_drop)
to_drop

['V50', 'V58', 'V64', 'V65', 'V67', 'V73', 'V83', 'V87', 'V90', 'V91']

# Random forest + f_classif

In [15]:
pipeline_rf1 = Pipeline([
    ('pre', SelectKBest(score_func=f_classif)),
    ('model', RandomForestClassifier(random_state=random_state))
])
    
param_grid_rf ={
    'pre__k': range(10,101,5),
    'model__n_estimators': range(50,401,50)
}

rf_1 = GridSearchCV(
    estimator=pipeline_rf1,
    param_grid=param_grid_rf,
    n_jobs=-1,
    cv=5,
    scoring='balanced_accuracy'
)
rf_1.fit(X_train,y_train)

,estimator,Pipeline(step...ate=111111))])
,param_grid,"{'model__n_estimators': range(50, 401, 50), 'pre__k': range(10, 101, 5)}"
,scoring,'balanced_accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,score_func,<function f_c...001E9F2761800>


In [16]:
support = rf_1.best_estimator_.named_steps['pre'].get_support()
features = np.array(X_train.columns)
print(features[support])

['V3' 'V8' 'V9' 'V10' 'V42' 'V51' 'V58' 'V65' 'V67' 'V91']


# Bagging + f_classif

In [19]:
pipeline_bag1 = Pipeline([
    ('pre', SelectKBest(score_func=f_classif)),
    ('bagging', BaggingClassifier(estimator=KNeighborsClassifier(), random_state=random_state))
])

param_grid_bag1 = {
    'pre__k': range(10,101,5),
    'bagging__n_estimators': range(5,30)
}

bag_grid1 = GridSearchCV(
    param_grid=param_grid_bag1,
    estimator=pipeline_bag1,
    n_jobs=-1,
    scoring='balanced_accuracy',
    cv=5
)

bag_grid1.fit(X_train,y_train)

,estimator,Pipeline(step...ate=111111))])
,param_grid,"{'bagging__n_estimators': range(5, 30), 'pre__k': range(10, 101, 5)}"
,scoring,'balanced_accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,score_func,<function f_c...001E9F2761800>


In [20]:
support = bag_grid1.best_estimator_.named_steps['pre'].get_support()
features = np.array(X_train.columns)
print(features[support])

['V2' 'V3' 'V8' 'V9' 'V10' 'V12' 'V16' 'V22' 'V25' 'V26' 'V36' 'V42' 'V48'
 'V51' 'V58' 'V64' 'V65' 'V67' 'V68' 'V72' 'V77' 'V83' 'V84' 'V91' 'V99']


# CHI2

In [26]:
from sklearn.feature_selection import chi2

selector = SelectKBest(score_func=chi2, k=10)

# 2. Obliczenie statystyk Chi-kwadrat i wybór najlepszych cech
selector.fit(X_train, y_train)

# (Opcjonalnie) Przetworzenie danych - zostawia tylko 10 wybranych kolumn
X_train_selected = selector.transform(X_train)
top_10_features = X_train.columns[selector.get_support()]
print("Top 10 zmiennych:", top_10_features.tolist())

Top 10 zmiennych: ['V3', 'V9', 'V10', 'V22', 'V42', 'V51', 'V64', 'V65', 'V83', 'V91']


In [27]:
wyniki = pd.DataFrame({
    'Cecha': X_train.columns,
    'Chi2_Score': selector.scores_,
    'p_value': selector.pvalues_
})

# Sortujemy malejąco po wyniku Chi2 i wyświetlamy 10 najlepszych
top_10_tabela = wyniki.sort_values(by='Chi2_Score', ascending=False).head(10)
print(top_10_tabela)

   Cecha  Chi2_Score        p_value
41   V42  952.151357  4.519516e-209
8     V9  804.221710  6.519112e-177
2     V3  766.038642  1.306388e-168
50   V51  568.260856  1.341616e-125
64   V65  557.600550  2.796289e-123
90   V91  394.842060   7.307545e-88
82   V83  287.589314   1.666596e-64
63   V64  128.937395   6.999157e-30
21   V22  110.589588   7.278081e-26
9    V10  104.093192   1.930203e-24
